# B2-022 — Probabilistic Latent Models

*30 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F4-matrices`, `book1:F3-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C6-neural-networks`, `book1:C7-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C8-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../book1/units/01-F1-scientific-python/lesson.ipynb), [book1:F4-matrices](../../../book1/units/04-F4-matrices/lesson.ipynb), [book1:F3-multivar-calculus](../../../book1/units/03-F3-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../book1/units/05-F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../book1/units/12-F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../book1/units/06-C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../book1/units/07-C2-linear-models/lesson.ipynb), [book1:C6-neural-networks](../../../book1/units/11-C6-neural-networks/lesson.ipynb), [book1:C7-pytorch](../../../book1/units/13-C7-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../book1/units/16-C9-dimensionality-reduction/lesson.ipynb), [book1:C8-neural-training](../../../book1/units/15-C8-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../B2-021-cross-modal-transformers-vision/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

This double-length unit builds the probabilistic toolkit behind latent-variable generative models: multivariate Gaussians, KL divergence, autoencoders, the reparameterization trick, and the variational autoencoder (VAE).
All models are tiny multilayer perceptrons trained on CPU from fixed seeds.

## Learning goals

- Evaluate multivariate Gaussian log-densities with `torch.linalg.slogdet` and `torch.linalg.solve`, read covariance ellipses from eigenpairs, sample with Cholesky or spectral factors, and fit Gaussians by maximum likelihood with stated existence conditions.
- Compute discrete KL with its zero conventions, prove nonnegativity from $\log x\le x-1$, contrast forward and reverse KL, and derive Gaussian KL closed forms.
- Build MLP autoencoders with an exact shape ledger, certify held-out reconstruction, and compare a linear autoencoder's subspace with PCA's by a projection-matrix distance.
- Derive the ELBO from Jensen's inequality, reparameterize $z=\mu+\sigma\odot\varepsilon$ with a log-variance head, and implement the per-example negative ELBO with explicit reductions.
- Train VAEs reproducibly, diagnose posterior collapse, apply $\beta$ weighting, sample from the prior, and interpolate in latent space.

## Owned concepts

| Concept | First taught |
|---|---|
| `multivariate-gaussian` | Session 1 |
| `kl-divergence` | Session 2 |
| `autoencoder` | Session 3 |
| `gaussian-reparameterization` | Session 4 |
| `variational-autoencoder` | Sessions 4–5 |

## Study route

0. [Book 1 bridge diagnostic](./lessons/00-book1-bridge.ipynb) — 30 minutes.
1. [Multivariate Gaussians](./lessons/01-multivariate-gaussians.ipynb) — 90 minutes. Then p01, p02, p06, p13.
2. [KL divergence](./lessons/02-kl-divergence.ipynb) — 90 minutes. Then p03, p07, p08, p14, p15.
3. [Autoencoders](./lessons/03-autoencoders.ipynb) — 90 minutes. Then p04, p09, p10, p17, p18.
4. [Reparameterization and the ELBO](./lessons/04-reparameterization-and-the-elbo.ipynb) — 90 minutes. Then p05, p11, p12, p16, p21.
5. [Training and using VAEs](./lessons/05-training-and-using-vaes.ipynb) — 90 minutes. Then p19, p20, p22, p23, p24.
6. [Review and self-quiz](./review.ipynb) — 60 minutes.

Practice sets: Set A (p01–p05) is retrieval multiple choice; Set B (p06–p16) is constrained coding and proof; Set C (p17–p24) is integrative training, scenario analysis, and challenge repair/audit.

## Tools taught inside this unit

Book 1 stops short of several tools this unit needs, so the lessons teach them before any practice uses them: determinants of symmetric positive-definite matrices (eigenvalue product, the $2\times2$ formula, and `slogdet`); the rule $\operatorname{Cov}(Az+b)=A\operatorname{Cov}(z)A^\top$; `torch.linalg.cholesky` as a stated-property API plus the spectral factor $Q\Lambda^{1/2}$; maximum likelihood for the mean and diagonal covariance (the full-covariance MLE is stated with its conditions); the tangent-line lemma $\log x\le x-1$ and Jensen's inequality for $\log$; the $0\cdot\log0$ convention; and the projection-matrix subspace distance.

## Reproducibility contract

The sole seed is `20260927`. The datasets `mixture2d`, `mixture8d`, and `lowrank8d` are produced by `scripts/generate_latent_data.py` and stored in `data/latent_datasets.json`; load them with `data/latent_data.py`, which verifies every row's SHA-256 and exposes immutable `TRAIN_IDS` and `HELDOUT_IDS` (row $i$ is held out exactly when $i \bmod 5=4$). Running `scripts/generate_latent_data.py --check` regenerates the rows and compares hashes. Training practices use CPU float32 tensors, full-batch Adam with the learning rate and step count stated in each statement, no shuffling, no schedulers, and no stored trained weights or final metrics.

## How to study

Run lessons in order, answer both checkpoints in every numbered section before reading its collected answers, and attempt each practice only after the session listed in the study route. Proofs require reasoning in full sentences; coding statements pin function names, shapes, dtypes, seeds, and tolerances, and every coding response ends with an answer-check section of executable asserts.

## Boundaries

MLP encoders and decoders only (no convolutions). No external data, torchvision datasets, model hubs, pretrained weights, GPU work, or downloads. GANs, diffusion, normalizing flows, and convolutional VAEs belong to `B2-023-generative-models-diffusion` or later and are named only as forward-only enrichment.

<!-- api-card -->
## Code in this unit

A quick reference for the library calls in this unit's code. Each one is explained in a comment (or in the text beside the code) where it first appears.

**New in this unit:**

- `torch.linalg.solve` — the `x` with `A x = b`
- `torch.linalg.eigvalsh` — the eigenvalues of a symmetric matrix, ascending
- `torch.linalg.slogdet` — `(sign, log|det A|)`, stable where `det A` would overflow
- `math.isclose` — True when two Python numbers differ by at most the tolerance
- `torch.linalg.cholesky` — the lower-triangular `L` with `L Lᵀ = Σ`
- `torch.linalg.eigh` — eigenvalues (ascending) and eigenvectors (columns) of a symmetric matrix
- `torch.diag` — the diagonal matrix with a vector on its diagonal (or a matrix's diagonal)
- `.sqrt` — the square root of each entry
- `torch.diagonal` — the main diagonal of a matrix, as a 1-D tensor
- `torch.log` — the natural log of each entry (`log 0` is `-inf`)
- `.exp` — `e` raised to each entry
- `torch.linalg.norm` — the norm of a tensor (Frobenius for a matrix by default)
- `F.mse_loss` — the mean squared error between two tensors
- `torch.linalg.svd` — the SVD `(U, S, Vh)`; `full_matrices=False` gives the thin version
- `.flatten` — all entries as one 1-D tensor
- `.sample` — a random draw from a distribution, with no gradient path
- `torch.distributions.Normal` — a normal distribution with given mean and standard deviation
- `.rsample` — a reparameterized draw `μ + σ ε`, so gradients reach `μ` and `σ`
- `.norm` — the Euclidean length of all entries
- `torch.normal` — one Gaussian draw per entry, with the given means and standard deviations
- `.dim` — the number of axes
- `torch.ones_like` — ones with the shape and dtype of the given tensor
- `torch.zeros_like` — zeros with the shape and dtype of the given tensor
- `np.linalg.slogdet` — `(sign, log|det A|)`, stable where `det A` would overflow
- `np.einsum` — sums products of entries following an index pattern, e.g. `"ki,ij,kj->"`
- `np.linalg.det` — the determinant of a square matrix
- `.register_forward_pre_hook` — calls a function with a module's inputs before each forward pass
- `.expand_as` — repeats length-1 axes to another tensor's shape, without copying
- `torch.cdist` — the `(n, m)` matrix of Euclidean distances between two sets of rows
- `torch.bincount` — how often each small non-negative integer occurs

**Also used here (met earlier):**

- `torch.exp` — `e` to each entry (first explained in B2-020)
- `torch.where` — pick by a condition (first explained in B2-021)
- `importlib.util.spec_from_file_location` — loading recipe for a Python file (first explained in B2-020)
- `importlib.util.module_from_spec` — empty module for a spec (first explained in B2-020)
- `AssertionError` — error raised by a failed check (first explained in Book 1 C12)